<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Aggregate and compare policy indicators across policy assessment runs.
- **Primary inputs**: runs/assessment_policies/assessment_*/<policy>/policies/indicator.csv
- **Primary outputs**: indicator_all_policies.csv, policy_summary_ap.csv, policy_summary_zp.csv
- **Refactor role**: Canonical notebook for portfolio-level policy KPIs. Keep; move shared indicator loading to shared/io_indicators.py.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Iterate over policy assessment runs and load each `policies/indicator.csv`.
2. Aggregate cost-effectiveness and CBA indicators across policies.
3. Export cross-policy summary tables.

### Inputs
- `runs/assessment_policies/assessment_<date_time>/<policy>/policies/indicator.csv`

### Outputs
- `indicator_all_policies.csv`
- `policy_summary_ap.csv`
- `policy_summary_zp.csv`

### How To Reuse
- Set the `folder` variable to the assessment run to summarize.


In [ ]:
import os
import pandas as pd

import matplotlib.pyplot as plt

from project.utils import reverse_dict
from project.input.resources import resources_data
from project.utils import make_stackedbar_plot

In [ ]:
# Refactor: assessment runs are under policy_assessment/runs/assessment_policies
folder = os.path.join('runs', 'assessment_policies', 'assessment_20240117_093354')

assert os.path.isdir(folder)

### Create indicator table with all policies

In [ ]:
variables = ['Cost effectiveness (euro/kWh)',
             'Cost effectiveness standard (euro/kWh)',
             'Cost effectiveness carbon (euro/tCO2)',
             'Leverage (%)',
             'Intensive margin insulation (%)',
             'Intensive margin insulation energy (%)',
             'Extensive margin insulation (%)',
             ]

# read result
result = {}
for f in [i for i in os.listdir(folder) if i != '.DS_Store']:
    try:
        df = pd.read_csv(os.path.join(folder, f, 'policies', 'indicator.csv'), index_col=[0])
        df = df.loc[[i for i in variables if i in df.index], :].dropna(axis=1, how='all')
        # df.columns = [int(i.split('-')[1]) for i in df.columns]
        result.update({f: df.T.to_dict()})
    except:
        pass
result = reverse_dict(result)

# format result
rslt = dict()
for key, item in result.items():
    rslt.update({key: pd.DataFrame(item).sort_index()})
rslt = pd.concat(rslt, axis=0)
rslt.index.names = ['Indicator', 'Year']

rslt.sort_index(axis=1, inplace=True)
rslt.to_csv(os.path.join(folder, 'indicator_all_policies.csv'))

rslt.reset_index(inplace=True)
# Step 1: Parse year to get the year number
rslt['Year_Number'] = rslt['Year'].str.extract('(\d+)').astype(int)

# Step 2: Keep information AP or ZP in another column
rslt['AP_ZP'] = rslt['Year'].str.extract('(AP|ZP)')

if 'carbon_tax' in rslt.columns:
    rslt.drop('carbon_tax', axis=1, inplace=True)


In [ ]:
def make_figure(rslt, key, grouped=None):

    # If key is a list, do shared y-axis subplot for indicators in the list
    if isinstance(key, list):
        # Create a figure with a shared y-axis
        fig, axes = plt.subplots(1, len(key), figsize=(14, 7), sharey=True)

        # Iterate over the axes and the indicators
        for ax, indicator in zip(axes, key):
            # Create a DataFrame with the data for the indicator
            df = rslt[rslt['Indicator'] == indicator]
            df = df[df['Year_Number'] != 1]
            df = df.drop(['Indicator', 'Year'], axis=1)

            # If grouped is not None, group the columns. If not in grouped, keep the column as is
            if grouped is not None:
                for column in df.columns:
                    if column not in grouped.keys():
                        grouped[column] = column

                df = df.T.groupby(grouped, dropna=False).first().T

            df.columns = [i.capitalize().replace('_', ' ') for i in df.columns]

            # Split the DataFrame into AP and ZP DataFrames
            df_ap = df[df['Ap zp'] == 'AP']
            df_zp = df[df['Ap zp'] == 'ZP']

            # Plot settings
            color_map = dict()

            # Placeholder for the legend entries
            color_legend_entries = []
            style_legend_entries = []

            unit = indicator.split('(')[-1].split(')')[0].replace('euro', '€')
            item = indicator.split(' (')[0].replace(' ', '_').lower()

            # Iterate over the columns to create a line plot for each
            for i, column in enumerate([i for i in df.columns if i not in ['Ap zp', 'Year number']]):
                name = column.capitalize().replace('_', ' ')
                # Define a color from the colormap
                color = plt.cm.tab10(i)
                color_map[column] = color

                # Plot AP - solid line with circle markers
                line1, = ax.plot(df_ap['Year number'], df_ap[column], label=f'{name}', linestyle='-', marker='o', color=color)
                # Plot ZP - dotted line with 'x' markers
                line2, = ax.plot(df_zp['Year number'], df_zp[column], label=f'{name}', linestyle=':', marker='x', color=color)

                # Add the first line of each pair to the color legend
                color_legend_entries.append(line1)
                # We only need one entry for AP and ZP styles, so we add them only once
                if i == 0:
                    style_legend_entries.extend([line1, line2])

            # set ax-title
            ax.set_title(indicator.split('(')[-0], fontsize=15)

            # fontsize to 12 for each axis
            ax.title.set_fontsize(15)
            ax.tick_params(which='both', bottom=False, top=False, labelsize=15)

            # remove spines top and right
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

        item = '_'.join([i.split(' (')[0].replace(' ', '_').lower() for i in key])

    else:
        # Create a DataFrame with the data for the indicator
        df = rslt[rslt['Indicator'] == key]
        df = df[df['Year_Number'] != 1]
        df = df.drop(['Indicator', 'Year'], axis=1)

        # If grouped is not None, group the columns. If not in grouped, keep the column as is
        if grouped is not None:
            for column in df.columns:
                if column not in grouped.keys():
                    grouped[column] = column

            df = df.T.groupby(grouped, dropna=False).first().T

        df.columns = [i.capitalize().replace('_', ' ') for i in df.columns]

        # Split the DataFrame into AP and ZP DataFrames
        df_ap = df[df['Ap zp'] == 'AP']
        df_zp = df[df['Ap zp'] == 'ZP']

        # Plot settings
        plt.figure(figsize=(14, 7))

        color_map = dict()

        # Placeholder for the legend entries
        color_legend_entries = []
        style_legend_entries = []

        unit = key.split('(')[-1].split(')')[0].replace('euro', '€')
        item = key.split(' (')[0].replace(' ', '_').lower()

        # Iterate over the columns to create a line plot for each
        for i, column in enumerate([i for i in df.columns if i not in ['Ap zp', 'Year number']]):
            name = column.capitalize().replace('_', ' ')
            # Define a color from the colormap
            color = plt.cm.tab10(i)
            color_map[column] = color

            # Plot AP - solid line with circle markers
            line1, = plt.plot(df_ap['Year number'], df_ap[column], label=f'{name}', linestyle='-', marker='o', color=color)
            # Plot ZP - dotted line with 'x' markers
            line2, = plt.plot(df_zp['Year number'], df_zp[column], label=f'{name}', linestyle=':', marker='x', color=color)

            # Add the first line of each pair to the color legend
            color_legend_entries.append(line1)
            # We only need one entry for AP and ZP styles, so we add them only once
            if i == 0:
                style_legend_entries.extend([line1, line2])


        # Adding title (and align to the left)
        plt.title(key.split('(')[-0])
        plt.gca().title.set_position([0.2, 1.05])
        plt.grid(False)

        # Remove axis tick ticks
        plt.tick_params(which='both', bottom=False, top=False, labelsize=15)

        # Remove top and right spines
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)


    # Set y-axis tick labels with "€/kWh" using FuncFormatter
    # Make rule to estimate the number of digits to display
    if unit == '%':
        plt.gca().yaxis.set_major_formatter(lambda y, _: '{:.0%}'.format(y))
    else:
        if df.loc[:, [i for i in df.columns if i not in ['Ap zp', 'Year number']]].max().max() > 10:
            plt.gca().yaxis.set_major_formatter(lambda y, _: '{:.0f} {}'.format(y, unit))
        elif df.loc[:, [i for i in df.columns if i not in ['Ap zp', 'Year number']]].max().max() > 0.6:
            plt.gca().yaxis.set_major_formatter(lambda y, _: '{:.1f} {}'.format(y, unit))
        else:
            plt.gca().yaxis.set_major_formatter(lambda y, _: '{:.2f} {}'.format(y, unit))


    # Add legends: one for colors and one for line styles, use font size 12 for text and title
    color_legend = plt.legend(handles=color_legend_entries, loc='upper left', title='Incentives', bbox_to_anchor=(1.05, 1),
                              frameon=False, fontsize=15, title_fontsize=15)
    # set style legend in black

    style_legend = plt.legend(handles=style_legend_entries, loc='lower left', title='Method', labels=['With interaction', 'Without interaction'],
                              handlelength=4, bbox_to_anchor=(1.05, 0.1),
                              handler_map={line1: plt.Line2D([], [], color='black', linestyle='-', marker='o'),
                                           line2: plt.Line2D([], [], color='black', linestyle=':', marker='x')},
                              frameon=False, fontsize=15, title_fontsize=15)



    # Add the first legend manually to the current Axes.
    plt.gca().add_artist(color_legend)

    # Put y_axis min to 0
    plt.ylim(bottom=0)

    # Ensure the plot is not clipped by the legends
    plt.tight_layout(rect=[0, 0, 0.95, 1])

    plt.savefig(os.path.join(folder, '{}.png'.format(item)))


In [ ]:
grouped = {
    'mpr_multifamily': 'multi-family',
    'mpr_multifamily_updated-mpr_multifamily_deep': 'multi-family',
    'cite': 'single-family',
    'mpr': 'single-family',
    'mpr_efficacite': 'single-family',
    'mpr_serenite': 'single-family_deep_renovation',
    'mpr_performance': 'single-family_deep_renovation',
    'cee': 'white_certificate',
    'reduced_vat': 'reduced_vat',
    'zero_interest_loan': 'zero_interest_loan'
}

order = ['multi-family', 'single-family', 'deep_renovation', 'white_certificate', 'reduced_vat']

indicator = {
    'Cost effectiveness (euro/kWh)': 'cost_effectiveness',
    'Cost effectiveness standard (euro/kWh)': 'cost_effectiveness_std',
    'Cost effectiveness carbon (euro/tCO2)': 'cost_effectiveness_carbon',
    'Leverage (%)': 'leverage',
    'Intensive margin insulation energy (%)': 'intensive_margin_insulation_energy',
    'Intensive margin insulation (%)': 'intensive_margin_insulation',
    'Extensive margin insulation (%)': 'extensive_margin_insulation'
 }
for key, item in indicator.items():
    make_figure(rslt, key, grouped=grouped)
make_figure(rslt, ['Cost effectiveness (euro/kWh)', 'Cost effectiveness standard (euro/kWh)'], grouped=grouped)
make_figure(rslt, ['Extensive margin insulation (%)', 'Intensive margin insulation energy (%)'], grouped=grouped)


### Create NPV indicator table with all policies

In [ ]:
variables = ['Investment',
             'Embodied emission',
             'Opportunity cost',
             'Energy saving',
             'Comfort EE',
             'Comfort prices',
             'Emission saving',
             'Health cost']

order = ['Subsidies', 'White certificate', 'Zero interest', 'Carbon tax', 'Mandatory']

for key in ['AP-1', 'ZP+1']:
    result = {}
    for f in [i for i in os.listdir(folder) if i != '.DS_Store' and os.path.isdir(os.path.join(folder, i))]:
        try:
            df = pd.read_csv(os.path.join(folder, f, 'policies', 'indicator.csv'), index_col=[0])
            df = df.loc[variables, key]
            name = f.capitalize().replace('_', ' ')
            result.update({name: df})
        except:
            pass
    result = pd.concat(result, axis=1)
    result = - result
    try:
        result = result[order]
    except KeyError:
        pass

    # policies = ['Cee', 'Mpr efficacite', 'Mpr performance', 'Zero interest loan', 'Obligation']
    # policies = ['Cee', 'Mpr', 'Reduced vta', 'Obligation']
    # result = result.loc[:, policies]
    if key == 'AP-1':
        title = 'Cost-benefits analysis with policy interactions (Billion euro)'
    elif key == 'ZP+1':
        title = 'Cost-benefits analysis without policy interaction (Billion euro)'
    else:
        title = 'Cost-benefits analysis (Billion euro)'

    make_stackedbar_plot(result.T, title, ncol=3, ymin=None,
                         format_y=lambda y, _: '{:.0f} B€'.format(y),
                         hline=0, colors=resources_data['colors'],
                         scatterplot=result.sum(),
                         save=os.path.join(folder, 'cost_benefits_analysis_{}.png'.format(key)),
                         rotation=0, left=1.3, fontxtick=15)

In [ ]:
result

### Create table for policies indicators

In [ ]:
variables = [
    'Consumption saving (TWh)',
    'Consumption saving (%)',
    'Consumption standard saving (TWh)',
    'Consumption standard saving (kWh/m2)',
    'Consumption standard saving (%)',
    'Emission saving (MtCO2)',
    'Emission saving (%)',
    'Cumulated emission saving (MtCO2)',
    'Energy poverty diff (Million)',
    'Energy poverty diff (%)',
    'Investment heater diff (Billion euro)',
    'Investment heater diff (%)',
    'Subsidies heater diff (Billion euro)',
    'Subsidies heater diff (%)',
    'Investment insulation diff (Billion euro)',
    'Investment insulation diff (%)',
    'Subsidies insulation diff (Billion euro)',
    'Subsidies insulation diff (%)'
]

result = {}
for key in ['AP-1', 'ZP+1']:
    temp = {}
    for f in [i for i in os.listdir(folder) if i != '.DS_Store' and os.path.isdir(os.path.join(folder, i))]:
        df = pd.read_csv(os.path.join(folder, f, 'policies', 'indicator.csv'), index_col=[0])
        df = df.loc[variables, key]
        name = f.capitalize().replace('_', ' ')
        temp.update({name: df})

    temp = pd.concat(temp, axis=1)
    temp.loc['Performance gap', :] = temp.loc['Consumption saving (TWh)', :] / temp.loc['Consumption standard saving (TWh)', :]
    result.update({key: temp})
result['AP-1'].to_csv(os.path.join(folder, 'policy_summary_ap.csv'))
result['ZP+1'].to_csv(os.path.join(folder, 'policy_summary_zp.csv'))

In [ ]:
result['AP-1']